In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms , datasets

from torch.utils.data import DataLoader
import torch.nn.functional as F
import os, random, shutil

In [3]:
tranform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.Grayscale(num_output_channels=1), # แปลงเป็นเทา
    transforms.RandomAffine(0, translate=(0.1,0.1)),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    # transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])  # Normalize(mean, std) | มาจาก : new = ((old−mean​)/std) มี 3 ตัวเพราะเป็น RGB
    transforms.Normalize([0.5], [0.5])
])
train_dir = "../../data/processed/Number/train"
val_dir = "../../data/processed/Number/val"
train_dataset = datasets.ImageFolder(root = train_dir , transform= tranform)
val_dataset = datasets.ImageFolder(root = val_dir , transform= tranform)

train_loader = DataLoader(train_dataset , batch_size = 64 , shuffle = True) # batch_size=32 | โมเดลจะเห็นทีละ 32 รูป
val_loader = DataLoader(val_dataset , batch_size = 64)

In [4]:
class Numclassification(nn.Module):
    def __init__(self):
        super(Numclassification , self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1 , out_channels=16 , kernel_size=3) # ภาพเข้า 1 ภาพ 3 สี , เเยกภาพวิเคราะห์ออกเป็น 16 ส่วน , คูณเมททริก something ยิงเลขเยอะยิ่งช้า
        self.conv2 = nn.Conv2d(in_channels=16 , out_channels=32 , kernel_size=3)
        self.conv3 = nn.Conv2d(32, 64, 3)
        
        self.bn1 = nn.BatchNorm2d(16)
        self.bn2 = nn.BatchNorm2d(32)
        self.bn3 = nn.BatchNorm2d(64)
        
        self.pool = nn.MaxPool2d(2,2) # ลดขนาดภาพ แต่ยังเก็บข้อมูลสำคัญไว้ ทำให้ train เร็วขึ้น
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(0.3)
        # self.fc1 = nn.Linear(32*14*14, 128) # nn.Linear(อินพุต, เอาต์พุต) FC → เอา feature ไป “ตัดสินใจ”
        # self.fc2 = nn.Linear(128, 10) # เป็นเลข 10 เพราะมี 10 class
        
        self.fc1 = nn.Linear(64*6*6, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        
    def forward(self, x):
        # x = self.conv1(x)   # 1. convolution หา feature เช่น เส้น / ขอบ
        # x = F.relu(x)       # 2. activation ตัดค่าติดลบออก (ทำให้ model เรียนรู้ nonlinear)
        # x = self.pool(x)    # 3. pooling ย่อภาพให้เล็กลง
        
        # x = self.conv2(x)   
        # x = F.relu(x)       
        # x = self.pool(x)    
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        
        x = self.flatten(x)
        x = self.dropout(x)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [5]:
model = Numclassification()

criterion = nn.CrossEntropyLoss()   # สำหรับ classification

optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Numclassification(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=2304, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
)

In [6]:
import torch

best_val_acc = 0.0 # สำหรับเก็บค่าที่ดีที่สุด

for epoch in range(80):
    model.train()

    correct_train = 0
    total_train = 0
    running_train_loss = 0.0 # 🔥 เพิ่มตัวแปรเก็บ Loss ของ Train

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # forward
        outputs = model(images)
        loss = criterion(outputs, labels)

        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # เก็บสะสมค่า Loss
        running_train_loss += loss.item() * images.size(0)

        # คำนวณ train accuracy
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    # คำนวณเฉลี่ยของ Train ต่อ Epoch
    train_acc = 100 * correct_train / total_train
    train_loss = running_train_loss / total_train
    
    model.eval()
    correct_val = 0
    total_val = 0
    running_val_loss = 0.0 # 🔥 เพิ่มตัวแปรเก็บ Loss ของ Validation

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels) # 🔥 คำนวณ Loss ของ Validation ด้วย

            running_val_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    # คำนวณเฉลี่ยของ Validation ต่อ Epoch
    val_acc = 100 * correct_val / total_val
    val_loss = running_val_loss / total_val
    
    # พิมพ์ผลลัพธ์
    print(f"Epoch {epoch+1:02d}/80")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    
    # 🔥 บันทึก Model ที่ดีที่สุด
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        # torch.save(model.state_dict(), 'best_model.pth')
        print("⭐ อัปเดตและบันทึกโมเดลที่ดีที่สุด!")

    print("-" * 40)

Epoch 01/80
Train Loss: 2.3225 | Train Acc: 10.57%
Val Loss:   2.3018 | Val Acc:   12.57%
⭐ อัปเดตและบันทึกโมเดลที่ดีที่สุด!
----------------------------------------
Epoch 02/80
Train Loss: 2.2895 | Train Acc: 11.46%
Val Loss:   2.2973 | Val Acc:   11.38%
----------------------------------------
Epoch 03/80
Train Loss: 2.2631 | Train Acc: 15.18%
Val Loss:   2.2760 | Val Acc:   17.37%
⭐ อัปเดตและบันทึกโมเดลที่ดีที่สุด!
----------------------------------------
Epoch 04/80
Train Loss: 2.2339 | Train Acc: 16.82%
Val Loss:   2.2439 | Val Acc:   15.57%
----------------------------------------
Epoch 05/80
Train Loss: 2.1945 | Train Acc: 20.24%
Val Loss:   2.1887 | Val Acc:   21.56%
⭐ อัปเดตและบันทึกโมเดลที่ดีที่สุด!
----------------------------------------
Epoch 06/80
Train Loss: 2.1319 | Train Acc: 22.32%
Val Loss:   2.1277 | Val Acc:   22.75%
⭐ อัปเดตและบันทึกโมเดลที่ดีที่สุด!
----------------------------------------
Epoch 07/80
Train Loss: 2.1042 | Train Acc: 21.73%
Val Loss:   2.0524 | Va

In [7]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy: {100 * correct / total:.2f}%")

Accuracy: 92.22%


In [9]:
PATH = "Numclassification.pth"
torch.save(model.state_dict(), PATH)

print(f"โมเดลถูกบันทึกเรียบร้อยที่ไฟล์: {PATH}")

โมเดลถูกบันทึกเรียบร้อยที่ไฟล์: Numclassification.pth
